# Stage 3 — Experiment Tracking (Scenario 1)
**IRP Project | Local-only MLflow setup**

This notebook adapts the professor's **Scenario 1** pattern to our updated **Stage 2** retail inventory project.

## What this notebook does
- loads the Stage 2 train / validation / test parquet files
- rebuilds the three Stage 2 models:
  - Logistic Regression
  - Random Forest
  - XGBoost
- logs each run to **local MLflow**
- tracks parameters, metrics, confusion matrices, reports, and model artifacts
- selects the **best run by validation F1 macro**
- saves the best MLflow `run_id` into `run_id.txt` for Stage 4

## Scenario 1 setup
- **Tracking server:** none
- **Backend store:** local filesystem
- **Artifact store:** local filesystem
- **MLflow UI:** launched locally with `mlflow ui`


In [2]:
%pip install mlflow xgboost imbalanced-learn pyarrow seaborn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import warnings
import subprocess
import sys
from pathlib import Path

import mlflow
import mlflow.sklearn
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
print("Libraries loaded")


Libraries loaded


## 1. MLflow local setup

In Scenario 1, MLflow writes directly to the local `mlruns/` folder. No tracking server is required.


In [4]:
print(f"Tracking URI before setup: {mlflow.get_tracking_uri()}")

# Local-only setup
mlflow.set_tracking_uri("file:" + str(Path("mlruns").resolve()))
mlflow.set_experiment("inventory-risk-stage3-scenario1")

print(f"Tracking URI after setup:  {mlflow.get_tracking_uri()}")


Traceback (most recent call last):
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
            

Tracking URI before setup: sqlite:////Users/ljyr/Documents/IE/Term%202/1%20MLOPS%20Machine%20Learning%20Operations/IRP-Section1Group5/03-experiment-tracking/mlflow.db
Tracking URI after setup:  file:/Users/ljyr/Documents/IE/Term 2/1 MLOPS Machine Learning Operations/IRP-Section1Group5/03-experiment-tracking/mlruns


## 2. Load Stage 2 outputs

This notebook expects the Stage 2 notebook to have already exported:
- `data/train.parquet`
- `data/val.parquet`
- `data/test.parquet`


In [9]:
train = pd.read_parquet("../02-features-modelling/data/train.parquet")
val   = pd.read_parquet("../02-features-modelling/data/val.parquet")
test  = pd.read_parquet("../02-features-modelling/data/test.parquet")

print("Train shape:", train.shape)
print("Val shape:  ", val.shape)
print("Test shape: ", test.shape)
train.head(2)


Train shape: (53900, 30)
Val shape:   (12300, 30)
Test shape:  (6000, 30)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,Inventory_Change,Inventory_Change_Pct,Days_of_Stock,Inventory_vs_Rolling7,Sales_Velocity,Coverage_Ratio,Forecast_Error,Order_to_Inventory,Risk_Label_Current,Risk_Label
0,2022-01-08,S001,P0001,Furniture,East,231,2,119,0.84,66.30,...,-1.0,-0.001965,254.000000,173.857143,0.003937,508.000000,1.16,0.234252,Safe Zone,Safe Zone
1,2022-01-09,S001,P0001,Electronics,South,373,350,178,352.24,41.72,...,117.0,0.230315,1.785714,251.285714,0.560000,1.774358,-2.24,0.284800,Safe Zone,Safe Zone


## 3. Recreate the Stage 2 feature setup


In [10]:
TARGET = "Risk_Label"

# Same leakage / non-model columns used in Stage 2
DROP_COLS = [
    "Risk_Label",
    "Risk_Label_Current",
    "Store ID",
    "Product ID",
    "Date",
    "Demand Forecast",
    "Demand_Forecast_Clean",
]

X_train = train.drop(columns=DROP_COLS).copy()
X_val = val.drop(columns=DROP_COLS).copy()
X_test = test.drop(columns=DROP_COLS).copy()

y_train_raw = train[TARGET].copy()
y_val_raw = val[TARGET].copy()
y_test_raw = test[TARGET].copy()

numerical_features = [
    "Inventory_Reconstructed",
    "Units Sold",
    "Units Ordered",
    "Price",
    "Discount",
    "Units_Sold_Lag1",
    "Inventory_Change_Pct",
    "Days_of_Stock",
    "Sales_Velocity",
    "Coverage_Ratio",
    "Forecast_Error",
    "Order_to_Inventory",
]

categorical_features = [
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
]

print("X_train shape:", X_train.shape)
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


X_train shape: (53900, 23)
Numerical features: ['Inventory_Reconstructed', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Units_Sold_Lag1', 'Inventory_Change_Pct', 'Days_of_Stock', 'Sales_Velocity', 'Coverage_Ratio', 'Forecast_Error', 'Order_to_Inventory']
Categorical features: ['Category', 'Region', 'Weather Condition', 'Seasonality']


In [11]:
# Match Stage 2 target encoding flow
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val = le.transform(y_val_raw)
y_test = le.transform(y_test_raw)

class_names = list(le.classes_)
all_labels = list(range(len(class_names)))

print("Encoded class order:", class_names)


Encoded class order: ['Overstock Risk', 'Safe Zone', 'Stockout Risk']


## 4. Build the same three models from Stage 2


In [12]:
# Logistic Regression pipeline
logit_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
])

logit_pipeline = Pipeline([
    ("preprocessor", logit_preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )),
])

# Random Forest pipeline
X_train_rf = X_train.copy()
X_val_rf = X_val.copy()
X_test_rf = X_test.copy()
for col in categorical_features:
    X_train_rf[col] = X_train_rf[col].astype("object")
    X_val_rf[col] = X_val_rf[col].astype("object")
    X_test_rf[col] = X_test_rf[col].astype("object")

cat_indices = [X_train_rf.columns.get_loc(col) for col in categorical_features]
smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)

rf_preprocessor = ColumnTransformer([
    ("num", "passthrough", numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

rf_pipeline = Pipeline([
    ("smote", smote_nc),
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )),
])

# XGBoost pipeline
xgb_preprocessor = ColumnTransformer([
    ("num", "passthrough", numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

xgb_pipeline = SkPipeline([
    ("preprocessor", xgb_preprocessor),
    ("model", XGBClassifier(
        objective="multi:softmax",
        num_class=3,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="mlogloss",
    )),
])

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
sample_weights = np.array([class_weight_dict[y] for y in y_train])

print("Pipelines ready")
print("XGBoost class weights:", class_weight_dict)


Pipelines ready
XGBoost class weights: {np.int64(0): np.float64(5.459333535906007), np.int64(1): np.float64(0.8282241583306443), np.int64(2): np.float64(0.6213399732558675)}


## 5. Training + evaluation helper


In [13]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def save_confusion_matrix(y_true, y_pred, split_name, model_name, out_dir):
    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    csv_path = out_dir / f"{split_name.lower()}_confusion_matrix.csv"
    fig_path = out_dir / f"{split_name.lower()}_confusion_matrix.png"

    cm_df.to_csv(csv_path)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{model_name} - {split_name} confusion matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150)
    plt.close()


def save_classification_report(y_true, y_pred, split_name, out_dir):
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=all_labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(out_dir / f"{split_name.lower()}_classification_report.csv")


def fit_and_predict(model_name, model):
    if model_name == "Random Forest":
        model.fit(X_train_rf, y_train)
        return {
            "train": model.predict(X_train_rf),
            "val": model.predict(X_val_rf),
            "test": model.predict(X_test_rf),
        }
    if model_name == "XGBoost":
        model.fit(X_train, y_train, model__sample_weight=sample_weights)
        return {
            "train": model.predict(X_train),
            "val": model.predict(X_val),
            "test": model.predict(X_test),
        }

    model.fit(X_train, y_train)
    return {
        "train": model.predict(X_train),
        "val": model.predict(X_val),
        "test": model.predict(X_test),
    }


## 6. Run MLflow experiments


In [14]:
models = {
    "Logistic Regression": logit_pipeline,
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline,
}

results = []
artifacts_root = Path("mlflow_artifacts")
artifacts_root.mkdir(exist_ok=True)

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name) as run:
        run_id = run.info.run_id
        run_dir = artifacts_root / run_id
        run_dir.mkdir(parents=True, exist_ok=True)

        mlflow.set_tag("scenario", "local-only")
        mlflow.set_tag("stage", "stage3")
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("project", "inventory-risk")

        mlflow.log_param("sklearn_version", sklearn.__version__)
        mlflow.log_param("train_rows", len(train))
        mlflow.log_param("val_rows", len(val))
        mlflow.log_param("test_rows", len(test))
        mlflow.log_param("n_numerical_features", len(numerical_features))
        mlflow.log_param("n_categorical_features", len(categorical_features))
        mlflow.log_param("target_name", TARGET)
        mlflow.log_param("class_names", json.dumps(class_names))

        if model_name == "Logistic Regression":
            mlflow.log_params({
                "max_iter": 2000,
                "class_weight": "balanced",
                "smote": True,
            })
        elif model_name == "Random Forest":
            mlflow.log_params({
                "n_estimators": 300,
                "max_depth": 10,
                "min_samples_leaf": 10,
                "max_features": "sqrt",
                "smotenc": True,
            })
        elif model_name == "XGBoost":
            mlflow.log_params({
                "objective": "multi:softmax",
                "num_class": 3,
                "n_estimators": 300,
                "max_depth": 6,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "sample_weighting": "balanced",
            })

        preds = fit_and_predict(model_name, model)

        split_targets = {
            "train": y_train,
            "val": y_val,
            "test": y_test,
        }

        summary = {"model": model_name, "run_id": run_id}

        for split_name, y_true in split_targets.items():
            y_pred = preds[split_name]
            metrics = compute_metrics(y_true, y_pred)
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(f"{split_name}_{metric_name}", metric_value)
                summary[f"{split_name}_{metric_name}"] = round(metric_value, 4)

            save_confusion_matrix(y_true, y_pred, split_name.upper(), model_name, run_dir)
            save_classification_report(y_true, y_pred, split_name.upper(), run_dir)

        mlflow.log_artifacts(str(run_dir), artifact_path="reports")
        mlflow.sklearn.log_model(model, artifact_path="model")

        results.append(summary)
        print(f"Logged {model_name} | run_id={run_id} | val_f1_macro={summary['val_f1_macro']:.4f}")

results_df = pd.DataFrame(results).sort_values(by="val_f1_macro", ascending=False).reset_index(drop=True)
results_df


2026/03/25 01:51:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 01:51:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged Logistic Regression | run_id=5e22a8a1e94547ae8432f514b4e8f48d | val_f1_macro=0.5088


2026/03/25 01:51:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 01:51:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged Random Forest | run_id=b0f3fa6beb454031991dea6f2516b472 | val_f1_macro=0.5243


2026/03/25 01:51:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/25 01:51:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged XGBoost | run_id=6d100de844854c29b0a323e7dd6d7a0d | val_f1_macro=0.5427


,model,run_id,train_accuracy,train_f1_macro,train_precision_macro,train_recall_macro,val_accuracy,val_f1_macro,val_precision_macro,val_recall_macro,test_accuracy,test_f1_macro,test_precision_macro,test_recall_macro
0,XGBoost,6d100de844854c29b0a323e7dd6d7a0d,0.7871,0.7198,0.6943,0.8253,0.6957,0.5427,0.5498,0.5576,0.7003,0.5451,0.5530,0.5596
1,Random Forest,b0f3fa6beb454031991dea6f2516b472,0.6842,0.5642,0.5822,0.6237,0.6629,0.5243,0.5449,0.5610,0.6578,0.5231,0.5470,0.5694
2,Logistic Regression,5e22a8a1e94547ae8432f514b4e8f48d,0.6314,0.5032,0.5476,0.5717,0.6465,0.5088,0.5477,0.5723,0.6387,0.4965,0.5398,0.5573


## 7. Save best run for Stage 4


In [15]:
best_row = results_df.iloc[0]
best_run_id = best_row["run_id"]
best_model_name = best_row["model"]

with open("run_id.txt", "w") as f:
    f.write(best_run_id)

print("Best model:", best_model_name)
print("Best run_id:", best_run_id)
print("Saved to run_id.txt")


Best model: XGBoost
Best run_id: 6d100de844854c29b0a323e7dd6d7a0d
Saved to run_id.txt


## 8. Launch the MLflow UI locally

Run the next cell once, then open:
- `http://127.0.0.1:5000`
- or `http://localhost:5000`


In [ ]:
subprocess.Popen([
    sys.executable,
    "-m", "mlflow", "ui",
    "--backend-store-uri", str(Path("mlruns").resolve()),
    "--host", "127.0.0.1",
    "--port", "5000",
])
print("MLflow UI started on http://127.0.0.1:5000")


MLflow UI started on http://127.0.0.1:5000


Registry store URI not provided. Using backend store URI.


/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/server/handlers.py:343: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, artifact_uri)
/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/server/handlers.py:378: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from oth

INFO:     127.0.0.1:51347 - "GET / HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:51347 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:51347 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:51348 - "GET /static-files/static/css/main.e1790ccd.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:51349 - "GET /static-files/static/js/main.c10baf8d.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:51347 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK


/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/server/handlers.py:343: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, artifact_uri)
Traceback (most recent call last):
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

INFO:     127.0.0.1:51348 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO:     127.0.0.1:51349 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK


/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/server/handlers.py:343: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, artifact_uri)
Traceback (most recent call last):
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

INFO:     127.0.0.1:51349 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
INFO:     127.0.0.1:51349 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC&filter=tags.%60mlflow.experiment.isGateway%60+IS+NULL HTTP/1.1" 200 OK


Traceback (most recent call last):
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ForML/lib/python3.11/site-packages/mlflow/store/tracking/file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
            

INFO:     127.0.0.1:51349 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC&filter=tags.%60mlflow.experiment.isGateway%60+IS+NULL HTTP/1.1" 200 OK
INFO:     127.0.0.1:51349 - "POST /graphql HTTP/1.1" 200 OK
INFO:     127.0.0.1:51349 - "POST /ajax-api/2.0/mlflow/experiments/search-datasets HTTP/1.1" 200 OK
INFO:     127.0.0.1:51349 - "GET /ajax-api/2.0/mlflow/gateway-proxy?gateway_path=api%2F2.0%2Fendpoints%2F HTTP/1.1" 200 OK
INFO:     127.0.0.1:51348 - "POST /ajax-api/2.0/mlflow/logged-models/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:51347 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:51352 - "POST /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
INFO:     127.0.0.1:51358 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK


## 9. Optional checks


In [ ]:
mlflow.search_experiments()
